# 🛡️ DeepGuardDB — Live Demo

This notebook lets you **upload any image** and run it through all 3 trained models to see if it's **Real** or **AI-Generated (Fake)**.

No retraining needed — models are loaded directly from saved checkpoints.

---
**Models:**
- ✅ Model A — Spatial CNN (88.15% accuracy)
- ✅ Model B — FFT CNN (76% accuracy)  
- ✅ Model C — FFT SVM (74% accuracy)

---
## 📋 Table of Contents
1. Setup & Load Models
2. 📈 Training History (Epochs)
3. 📊 Final Evaluation Results
4. 🌊 FFT Spectrum Comparisons
5. 🚀 Test Your Own Image
6. 📁 Batch Test Multiple Images

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 1: Imports & Setup
# ─────────────────────────────────────────────────────────
import sys
import warnings
import json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import joblib
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from PIL import Image
from pathlib import Path

# Add project root to path so we can import our modules
PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from model_cnn import build_spatial_cnn, build_fft_cnn
from preprocess import get_spatial_transforms, get_fft_transforms
from fft_features import extract_fft_features, features_to_vector, _image_to_magnitude

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LOGS_DIR    = PROJECT_ROOT / 'logs'
RESULTS_DIR = PROJECT_ROOT / 'results'
CHECKPOINTS = PROJECT_ROOT / 'checkpoints'

print(f'✅ Setup complete!')
print(f'   Device      : {DEVICE}')
print(f'   Project root: {PROJECT_ROOT}')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 2: Load All 3 Trained Models
# ─────────────────────────────────────────────────────────

# --- Model A: Spatial CNN ---
model_spatial = build_spatial_cnn()
model_spatial.load_state_dict(torch.load(CHECKPOINTS / 'cnn_spatial_best.pth', map_location=DEVICE))
model_spatial.to(DEVICE)
model_spatial.eval()
print('✅ Model A (Spatial CNN) loaded')

# --- Model B: FFT CNN ---
model_fft_cnn = build_fft_cnn()
model_fft_cnn.load_state_dict(torch.load(CHECKPOINTS / 'cnn_fft_best.pth', map_location=DEVICE))
model_fft_cnn.to(DEVICE)
model_fft_cnn.eval()
print('✅ Model B (FFT CNN) loaded')

# --- Model C: FFT SVM ---
svm_pipeline = joblib.load(CHECKPOINTS / 'svm_fft_best.pkl')
print('✅ Model C (FFT SVM) loaded')

print('\n🎯 All models ready!')

---
## 📈 Training History — Epoch by Epoch

This shows exactly what happened **during training** — how the loss dropped and accuracy climbed over 20 epochs.

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 3: Training History — Loss & Accuracy Curves
# ─────────────────────────────────────────────────────────

# Load both training logs
df_spatial = pd.read_csv(LOGS_DIR / 'cnn_spatial_metrics.csv')
df_fft     = pd.read_csv(LOGS_DIR / 'cnn_fft_metrics.csv')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('📈 Training History — All 20 Epochs', 
             fontsize=16, color='white', fontweight='bold', y=0.98)

epochs = df_spatial['epoch']

plot_cfg = [
    # (ax, df, y_train_col, y_val_col, title, ylabel, train_color, val_color)
    (axes[0,0], df_spatial, 'train_loss', 'val_loss',
     '🧠 Spatial CNN — Loss per Epoch', 'Loss',
     '#4fc3f7', '#ff8a65'),
    (axes[0,1], df_spatial, 'train_acc',  'val_acc',
     '🧠 Spatial CNN — Accuracy per Epoch', 'Accuracy',
     '#4fc3f7', '#ff8a65'),
    (axes[1,0], df_fft,     'train_loss', 'val_loss',
     '🌊 FFT CNN — Loss per Epoch', 'Loss',
     '#a78bfa', '#34d399'),
    (axes[1,1], df_fft,     'train_acc',  'val_acc',
     '🌊 FFT CNN — Accuracy per Epoch', 'Accuracy',
     '#a78bfa', '#34d399'),
]

for ax, df, train_col, val_col, title, ylabel, tc, vc in plot_cfg:
    ax.set_facecolor('#16213e')

    ax.plot(df['epoch'], df[train_col], color=tc, linewidth=2.5,
            marker='o', markersize=5, label='Train')
    ax.plot(df['epoch'], df[val_col],   color=vc, linewidth=2.5,
            marker='s', markersize=5, label='Validation', linestyle='--')

    # Highlight best val epoch
    if 'acc' in val_col:
        best_idx = df[val_col].idxmax()
        best_val = df[val_col].max()
        label_text = f'Best: {best_val:.4f}'
    else:
        best_idx = df[val_col].idxmin()
        best_val = df[val_col].min()
        label_text = f'Best: {best_val:.4f}'

    best_epoch = df['epoch'].iloc[best_idx]
    ax.axvline(x=best_epoch, color='gold', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.annotate(f'  Epoch {best_epoch}\n  {label_text}',
                xy=(best_epoch, best_val),
                color='gold', fontsize=9,
                xytext=(best_epoch + 0.3, best_val))

    ax.set_title(title, color='white', fontsize=12, pad=8)
    ax.set_xlabel('Epoch', color='white')
    ax.set_ylabel(ylabel, color='white')
    ax.tick_params(colors='white')
    ax.spines[:].set_color('#444')
    ax.legend(facecolor='#16213e', labelcolor='white', fontsize=10)
    ax.set_xticks(df['epoch'])

plt.tight_layout()
plt.show()

# ── Epoch-by-epoch table ────────────────────────────────
print('\n━' * 35)
print('  🧠 SPATIAL CNN — Epoch Summary')
print('━' * 35)
print(f'  {"Epoch":<7} {"Train Loss":<13} {"Train Acc":<13} {"Val Loss":<13} {"Val Acc"}')
print('  ' + '-' * 60)
for _, row in df_spatial.iterrows():
    marker = ' ⭐' if row['val_acc'] == df_spatial['val_acc'].max() else ''
    print(f'  {int(row["epoch"]):<7} {row["train_loss"]:<13.4f} {row["train_acc"]:<13.4f} {row["val_loss"]:<13.4f} {row["val_acc"]:.4f}{marker}')

print('\n━' * 35)
print('  🌊 FFT CNN — Epoch Summary')
print('━' * 35)
print(f'  {"Epoch":<7} {"Train Loss":<13} {"Train Acc":<13} {"Val Loss":<13} {"Val Acc"}')
print('  ' + '-' * 60)
for _, row in df_fft.iterrows():
    marker = ' ⭐' if row['val_acc'] == df_fft['val_acc'].max() else ''
    print(f'  {int(row["epoch"]):<7} {row["train_loss"]:<13.4f} {row["train_acc"]:<13.4f} {row["val_loss"]:<13.4f} {row["val_acc"]:.4f}{marker}')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 4: Training Speed — Time per Epoch
# ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('⏱️ Training Speed — Seconds per Epoch', 
             fontsize=14, color='white', fontweight='bold')

for ax, df, label, color in [
    (axes[0], df_spatial, 'Spatial CNN', '#4fc3f7'),
    (axes[1], df_fft,     'FFT CNN',     '#a78bfa'),
]:
    ax.set_facecolor('#16213e')
    ax.bar(df['epoch'], df['elapsed_s'], color=color, alpha=0.85, edgecolor='white', linewidth=0.4)
    ax.axhline(y=df['elapsed_s'].mean(), color='gold', linestyle='--', linewidth=1.5,
               label=f'Avg: {df["elapsed_s"].mean():.1f}s')
    ax.set_title(f'{label}', color='white', fontsize=12, pad=8)
    ax.set_xlabel('Epoch', color='white')
    ax.set_ylabel('Seconds', color='white')
    ax.tick_params(colors='white')
    ax.spines[:].set_color('#444')
    ax.legend(facecolor='#16213e', labelcolor='white')

plt.tight_layout()
plt.show()

total_spatial = df_spatial['elapsed_s'].sum()
total_fft     = df_fft['elapsed_s'].sum()
print(f'⏱️  Spatial CNN total training time : {total_spatial:.0f}s ({total_spatial/60:.1f} min)')
print(f'⏱️  FFT CNN total training time     : {total_fft:.0f}s ({total_fft/60:.1f} min)')

---
## 📊 Final Evaluation Results

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 5: Display Saved Evaluation Charts
# ─────────────────────────────────────────────────────────

with open(RESULTS_DIR / 'evaluation_summary.json') as f:
    summary = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('📊 Final Evaluation Results — DeepGuardDB', 
             fontsize=15, color='white', fontweight='bold')

charts = [
    ('accuracy_comparison.png', 'Accuracy Comparison'),
    ('roc_curves.png', 'ROC Curves'),
    ('cm_cnn_spatial.png', 'Confusion Matrix — Spatial CNN'),
]

for ax, (fname, title) in zip(axes, charts):
    fpath = RESULTS_DIR / fname
    if fpath.exists():
        ax.imshow(plt.imread(str(fpath)))
        ax.set_title(title, color='white', fontsize=11, pad=6)
    else:
        ax.text(0.5, 0.5, f'Not found:\n{fname}', ha='center', va='center',
                color='red', transform=ax.transAxes)
    ax.axis('off')

plt.tight_layout()
plt.show()

print('\n📋 Final Test Set Results (1,072 unseen images):')
print(f'{"Model":<15} {"Accuracy":>10} {"ROC AUC":>10}')
print('-' * 37)
name_map = {'cnn_spatial': 'Spatial CNN', 'cnn_fft': 'FFT CNN', 'svm_fft': 'FFT SVM'}
for key, label in name_map.items():
    acc = summary[key]['accuracy'] * 100
    auc = summary[key]['roc_auc']
    print(f'{label:<15} {acc:>9.2f}% {auc:>10.4f}')

---
## 🌊 FFT Spectrum Comparisons

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 6: FFT Spectrum Visualizations from Test Set
# ─────────────────────────────────────────────────────────

VIZ_DIR  = RESULTS_DIR / 'visualizations'
fft_imgs = sorted(VIZ_DIR.glob('fft_comparison_*.png'))

if fft_imgs:
    for fpath in fft_imgs:
        fig, ax = plt.subplots(figsize=(14, 5))
        fig.patch.set_facecolor('#1a1a2e')
        ax.imshow(plt.imread(str(fpath)))
        ax.set_title(f'🌊 {fpath.name}', color='white', fontsize=12)
        ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print('No FFT visualizations found in results/visualizations/')

---
## 🚀 TEST YOUR OWN IMAGE

Change the path below to **any image** on your computer — the models will tell you if it's Real or AI-generated!

**Supported formats:** `.jpg`, `.jpeg`, `.png`, `.webp`, `.bmp`

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 7: Prediction Helper (run once)
# ─────────────────────────────────────────────────────────

spatial_transform = get_spatial_transforms('val')
fft_transform     = get_fft_transforms('val')
LABELS = ['REAL ✅', 'FAKE ❌']
COLORS = ['#2ecc71', '#e74c3c']

def predict_all(image_path: str):
    img_path = Path(image_path)
    if not img_path.exists():
        raise FileNotFoundError(f"Image not found: {img_path}")

    pil_img = Image.open(img_path).convert('RGB')
    img_np  = np.array(pil_img)
    results = {}

    with torch.no_grad():
        # Model A: Spatial CNN
        tensor_s = spatial_transform(pil_img).unsqueeze(0).to(DEVICE)
        probs_s  = torch.softmax(model_spatial(tensor_s), dim=1).cpu().numpy()[0]
        pred_s   = int(np.argmax(probs_s))
        results['Spatial CNN'] = {
            'label': LABELS[pred_s], 'pred': pred_s,
            'confidence': float(probs_s[pred_s]) * 100,
            'prob_real': float(probs_s[0]) * 100,
            'prob_fake': float(probs_s[1]) * 100,
            'color': COLORS[pred_s]
        }

        # Model B: FFT CNN
        tensor_f = fft_transform(pil_img).unsqueeze(0).to(DEVICE)
        probs_f  = torch.softmax(model_fft_cnn(tensor_f), dim=1).cpu().numpy()[0]
        pred_f   = int(np.argmax(probs_f))
        results['FFT CNN'] = {
            'label': LABELS[pred_f], 'pred': pred_f,
            'confidence': float(probs_f[pred_f]) * 100,
            'prob_real': float(probs_f[0]) * 100,
            'prob_fake': float(probs_f[1]) * 100,
            'color': COLORS[pred_f]
        }

    # Model C: FFT SVM
    img_bgr   = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
    feats     = extract_fft_features(img_bgr)
    vec       = features_to_vector(feats).reshape(1, -1)
    probs_svm = svm_pipeline.predict_proba(vec)[0]
    pred_svm  = int(np.argmax(probs_svm))
    results['FFT SVM'] = {
        'label': LABELS[pred_svm], 'pred': pred_svm,
        'confidence': float(probs_svm[pred_svm]) * 100,
        'prob_real': float(probs_svm[0]) * 100,
        'prob_fake': float(probs_svm[1]) * 100,
        'color': COLORS[pred_svm]
    }
    return pil_img, img_np, results


def show_results(image_path: str):
    print(f'\n🔍 Analysing: {Path(image_path).name}...\n')
    pil_img, img_np, results = predict_all(image_path)

    img_bgr   = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
    gray      = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    magnitude = _image_to_magnitude(gray)
    mn, mx    = magnitude.min(), magnitude.max()
    mag_norm  = (magnitude - mn) / (mx - mn + 1e-12)

    fig = plt.figure(figsize=(18, 10))
    fig.patch.set_facecolor('#1a1a2e')
    fig.suptitle('🛡️  DeepGuardDB — Deepfake Detection Analysis',
                 fontsize=18, fontweight='bold', color='white', y=0.98)

    ax_img = fig.add_axes([0.02, 0.52, 0.28, 0.40])
    ax_img.imshow(pil_img)
    ax_img.set_title('Input Image', color='white', fontsize=12, pad=8)
    ax_img.axis('off')

    ax_fft = fig.add_axes([0.32, 0.52, 0.28, 0.40])
    ax_fft.imshow(mag_norm, cmap='inferno')
    ax_fft.set_title('FFT Frequency Spectrum', color='white', fontsize=12, pad=8)
    ax_fft.axis('off')

    models  = list(results.keys())
    ax_bar  = fig.add_axes([0.63, 0.52, 0.34, 0.40])
    ax_bar.set_facecolor('#16213e')
    y_pos   = np.arange(len(models))
    confs   = [results[m]['confidence'] for m in models]
    bcolors = [results[m]['color'] for m in models]
    bars    = ax_bar.barh(y_pos, confs, color=bcolors, height=0.5,
                          edgecolor='white', linewidth=0.5)
    for i, (bar, model) in enumerate(zip(bars, models)):
        c = results[model]['confidence']
        ax_bar.text(c + 1, i, f"{c:.1f}%  {results[model]['label']}",
                    va='center', color='white', fontsize=11, fontweight='bold')
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(models, color='white', fontsize=11)
    ax_bar.set_xlim(0, 130)
    ax_bar.set_xlabel('Confidence %', color='white')
    ax_bar.set_title('Model Predictions', color='white', fontsize=12, pad=8)
    ax_bar.tick_params(colors='white')
    ax_bar.spines[:].set_color('#444')
    ax_bar.xaxis.label.set_color('white')

    for i, model in enumerate(models):
        ax = fig.add_axes([0.02 + i * 0.33, 0.06, 0.28, 0.38])
        ax.set_facecolor('#16213e')
        rp = results[model]['prob_real']
        fp = results[model]['prob_fake']
        b2 = ax.bar(['Real', 'Fake'], [rp, fp],
                    color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=0.5, width=0.5)
        for bar2, val in zip(b2, [rp, fp]):
            ax.text(bar2.get_x() + bar2.get_width()/2, bar2.get_height() + 1,
                    f'{val:.1f}%', ha='center', va='bottom',
                    color='white', fontsize=12, fontweight='bold')
        ax.set_title(f'{model}\n→ {results[model]["label"]}',
                     color=results[model]['color'], fontsize=11, fontweight='bold', pad=8)
        ax.set_ylim(0, 115)
        ax.set_ylabel('Probability %', color='white', fontsize=9)
        ax.tick_params(colors='white')
        ax.spines[:].set_color('#444')
        ax.yaxis.label.set_color('white')

    plt.show()

    print('━' * 55)
    print(f'  📊 RESULTS: {Path(image_path).name}')
    print('━' * 55)
    for model, r in results.items():
        print(f'  {model:<14} → {r["label"]}  ({r["confidence"]:.1f}% confident)')
    print('━' * 55)
    votes    = [r['pred'] for r in results.values()]
    majority = 'FAKE ❌' if sum(votes) >= 2 else 'REAL ✅'
    print(f'  🗳️  MAJORITY VOTE  → {majority}')
    print('━' * 55)

print('✅ All functions ready!')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 8: ⬇️  PUT YOUR IMAGE PATH HERE
# ─────────────────────────────────────────────────────────

IMAGE_PATH = r"C:\path\to\your\image.jpg"   # ← Change this!

show_results(IMAGE_PATH)

---
## 📁 Batch Test: Try Multiple Images at Once

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 9: Batch Test — Multiple Images
# ─────────────────────────────────────────────────────────

IMAGE_LIST = [
    r"C:\path\to\real_photo.jpg",      # ← Replace with your images
    r"C:\path\to\ai_generated.jpg",
]

print('\n📋 BATCH TEST RESULTS')
print('=' * 70)
print(f'{"Image":<30} {"Spatial CNN":<20} {"FFT CNN":<20} {"FFT SVM"}')
print('-' * 70)

for img_path in IMAGE_LIST:
    try:
        _, _, results = predict_all(img_path)
        name = Path(img_path).name[:28]
        row  = f'{name:<30}'
        for model in ['Spatial CNN', 'FFT CNN', 'FFT SVM']:
            r    = results[model]
            icon = '✅' if r['pred'] == 0 else '❌'
            row += f" {icon} {r['confidence']:5.1f}%        "
        print(row)
    except FileNotFoundError as e:
        print(f'  ⚠️  {e}')

print('=' * 70)